# Importar ficheiros INF001 (Informe horas entrega) para DuckDB

1. Preparar as tabelas `entregas_2025` / `entregas_2026`
2. Ler os ficheiros `.xls` Excel XML Spreadsheet 2003
3. Inserir sem duplicar linhas
4. Validar os ficheiros contra a base de dados


## Configuracao

In [1]:
# ==========================
# 1. Configuração
# ==========================

import platform
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path

import duckdb
import pandas as pd

if platform.system() == "Windows":
    DB_PATH = Path(
        r"C:\Users\LISARR\Documents\python\00.DB\2026.duckdb"
    )
    PASTA_FICHEIROS = Path(
        r"C:\Users\LISARR\Desktop\02.Pontualidade\entregas"
    )

elif platform.system() == "Darwin":
    DB_PATH = (
        Path.home()
        / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2026.duckdb"
    )
    PASTA_FICHEIROS = (
        Path.home()
        / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/entregas"
    )

else:
    raise OSError(
        f"Sistema operativo não suportado: {platform.system()}"
    )

DB_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"DB_PATH: {DB_PATH}")
print(f"PASTA_FICHEIROS: {PASTA_FICHEIROS}")


DB_PATH: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2026.duckdb
PASTA_FICHEIROS: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/entregas


In [2]:
# Colunas do ficheiro INF001: nome_sql -> nome original da coluna no Excel
COLUNAS_ENTREGAS = {
    "PROPIETARIO_DT": "Propietario DT",
    "RECORRIDO_CAMION": "Recorrido Cami\u00f3n",
    "DESTINO": "Destino",
    "HORA_TEORICA_LLEGADA": "Hora te\u00f3rica Llegada",
    "HORA_REAL_LLEGADA": "Hora Real Llegada",
    "HORA_SALIDA_DESTINO": "Hora Salida de Destino",
    "RETRASO_LLEGADA": "Retraso Llegada",
    "TIEMPO_ESPERA": "Tiempo de Espera",
    "NUM_UT": "N\u00fam. UT",
    "FECHA_TEORICA_LLEGADA": "Fecha Te\u00f3rica Llegada",
    "FECHA_REAL_LLEGADA": "Fecha Real Llegada",
    "TEMPERATURA": "Temperatura",
    "EMPRESA_TRANSPORTE": "Empresa de Transporte",
    "MATRICULA_REMOLQUE": "Matr\u00edcula Remolque",
    "MOTIVO": "Motivo",
    "OBSERVACIONES": "Observaciones",
}

COLUNA_ORIGEM = "ficheiro_origem"
ANOS = ("2025", "2026")

# Colunas (nomes sql) que identificam uma linha como unica. Usa TODAS
# as colunas - uma linha so e considerada duplicada se for igual em
# tudo. Usadas pelo indice unico para nunca deixar a mesma entrega ser
# inserida 2 vezes.
CHAVE_UNICA = tuple(COLUNAS_ENTREGAS.keys())

## Passo 1 — Preparar a base de dados

In [3]:
# ==========================
# 3. Estrutura DuckDB
# ==========================

def preparar_base_dados(con):
    colunas_sql = ",\n        ".join(
        f'"{coluna}" VARCHAR'
        for coluna in COLUNAS_ENTREGAS
    )

    for ano in ANOS:
        tabela = f"entregas_{ano}"

        con.execute(f"""
            CREATE TABLE IF NOT EXISTS "{tabela}" (
                id BIGINT,
                {colunas_sql},
                "{COLUNA_ORIGEM}" VARCHAR
            )
        """)

        # DuckDB permite várias linhas NULL num índice UNIQUE.
        # A importação abaixo elimina duplicados com comparação
        # NULL-safe antes de inserir.

        con.execute(
            f'CREATE INDEX IF NOT EXISTS '
            f'"idx_{tabela}_data" '
            f'ON "{tabela}" ("FECHA_TEORICA_LLEGADA")'
        )


with duckdb.connect(str(DB_PATH)) as con:
    preparar_base_dados(con)

print("Base de dados e tabelas prontas.")


Base de dados e tabelas prontas.


## Passo 2 — Ler os ficheiros da pasta

In [4]:
def listar_ficheiros_excel(pasta):
    """Procura ficheiros .xls (tambem em subpastas), sem duplicar por
    causa de maiusculas/minusculas no caminho."""
    encontrados = glob.glob(os.path.join(pasta, "**", "*.xls"), recursive=True)
    vistos = set()
    ficheiros = []
    for caminho in encontrados:
        chave = os.path.normcase(os.path.abspath(caminho))
        if chave not in vistos:
            vistos.add(chave)
            ficheiros.append(caminho)
    return sorted(ficheiros)


ficheiros = listar_ficheiros_excel(PASTA_FICHEIROS)
print(f"Ficheiros .xls encontrados: {len(ficheiros)}")

NameError: name 'glob' is not defined

In [ ]:
# ==========================
# 1. Leitura Excel XML
# ==========================
NS = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}
SS_INDEX = "{urn:schemas-microsoft-com:office:spreadsheet}Index"


def ler_linhas_xml_spreadsheet(caminho_ficheiro, min_colunas_cabecalho=5):
    """Le todas as worksheets validas de um Excel XML Spreadsheet 2003."""
    tree = ET.parse(caminho_ficheiro)
    root = tree.getroot()

    worksheets = root.findall("ss:Worksheet", NS)
    if not worksheets:
        raise ValueError("Nao foi encontrada nenhuma 'Worksheet' no ficheiro.")

    def extrair_linha(linha_xml):
        valores = []
        proximo_indice = 1

        for cell in linha_xml.findall("ss:Cell", NS):
            idx = cell.get(SS_INDEX)
            idx = int(idx) if idx is not None else proximo_indice

            while len(valores) < idx - 1:
                valores.append(None)

            data_el = cell.find("ss:Data", NS)
            valores.append(data_el.text if data_el is not None else None)
            proximo_indice = idx + 1

        return valores

    cabecalho_referencia = None
    linhas_totais = []

    for worksheet in worksheets:
        tabela = worksheet.find("ss:Table", NS)
        if tabela is None:
            continue

        linhas_xml = tabela.findall("ss:Row", NS)
        if not linhas_xml:
            continue

        idx_cabecalho = None
        cabecalho = None

        for i, linha_xml in enumerate(linhas_xml):
            valores = extrair_linha(linha_xml)
            preenchidas = sum(v not in (None, "") for v in valores)

            if preenchidas >= min_colunas_cabecalho:
                idx_cabecalho = i
                cabecalho = [
                    str(v).strip() if v else f"COLUNA_{j + 1}"
                    for j, v in enumerate(valores)
                ]
                break

        if idx_cabecalho is None:
            continue

        # So agrega folhas com o mesmo cabecalho de dados
        if cabecalho_referencia is None:
            cabecalho_referencia = cabecalho
        elif cabecalho != cabecalho_referencia:
            continue

        n_colunas = len(cabecalho_referencia)

        for linha_xml in linhas_xml[idx_cabecalho + 1:]:
            valores = extrair_linha(linha_xml)

            if len(valores) < n_colunas:
                valores += [None] * (n_colunas - len(valores))
            elif len(valores) > n_colunas:
                valores = valores[:n_colunas]

            linhas_totais.append(valores)

    if cabecalho_referencia is None:
        raise ValueError("Nao foi encontrada uma linha de cabecalho valida.")

    return cabecalho_referencia, linhas_totais


print("Funcao ler_linhas_xml_spreadsheet() pronta.")


## Passo 3 — Inserir os dados (sem duplicar)

In [ ]:
# ==========================
# 5. Preparação dos dados
# ==========================

def validar_data(valor):
    if not valor:
        return None

    texto = str(valor).strip()

    if (
        not texto
        or texto.upper() == "N/D"
    ):
        return None

    try:
        return datetime.strptime(
            texto,
            "%d/%m/%Y",
        )

    except ValueError:
        return None


def extrair_ano(mapa):
    valor = mapa.get(
        COLUNAS_ENTREGAS[
            "FECHA_TEORICA_LLEGADA"
        ]
    )

    data = validar_data(
        valor
    )

    return (
        str(data.year)
        if data
        else None
    )


def montar_linha(
    mapa,
    nome_ficheiro,
):
    valores = []

    for nome_original in (
        COLUNAS_ENTREGAS.values()
    ):
        valor = mapa.get(
            nome_original
        )

        if isinstance(
            valor,
            str,
        ):
            valor = valor.strip()

        valores.append(
            valor
        )

    return valores + [
        nome_ficheiro
    ]


def inserir_batch(
    con,
    ano,
    linhas,
):
    if not linhas:
        return 0, 0

    tabela = f"entregas_{ano}"

    colunas_dados = list(
        COLUNAS_ENTREGAS.keys()
    )

    colunas_batch = [
        *colunas_dados,
        COLUNA_ORIGEM,
    ]

    df_batch = pd.DataFrame(
        linhas,
        columns=colunas_batch,
        dtype=object,
    )

    # Duplicados dentro do próprio ficheiro/batch.
    antes_batch = len(
        df_batch
    )

    df_batch = df_batch.drop_duplicates(
        subset=colunas_dados,
        keep="first",
    )

    duplicados_batch = (
        antes_batch - len(df_batch)
    )

    if df_batch.empty:
        return 0, duplicados_batch

    max_id = con.execute(f"""
        SELECT COALESCE(
            MAX(TRY_CAST(id AS BIGINT)),
            0
        )
        FROM "{tabela}"
    """).fetchone()[0]

    con.register(
        "batch_entregas",
        df_batch,
    )

    condicao = " AND ".join(
        f't."{coluna}" '
        f'IS NOT DISTINCT FROM '
        f'b."{coluna}"'
        for coluna in colunas_dados
    )

    colunas_select = ",\n                ".join(
        f'b."{coluna}"'
        for coluna in colunas_dados
    )

    try:
        existentes = con.execute(f"""
            SELECT COUNT(*)
            FROM batch_entregas b
            WHERE EXISTS (
                SELECT 1
                FROM "{tabela}" t
                WHERE {condicao}
            )
        """).fetchone()[0]

        con.execute(f"""
            INSERT INTO "{tabela}"
            SELECT
                {int(max_id)}
                + ROW_NUMBER() OVER () AS id,
                {colunas_select},
                b."{COLUNA_ORIGEM}"
            FROM batch_entregas b
            WHERE NOT EXISTS (
                SELECT 1
                FROM "{tabela}" t
                WHERE {condicao}
            )
        """)

    finally:
        con.unregister(
            "batch_entregas"
        )

    inseridos = (
        len(df_batch) - existentes
    )

    duplicados = (
        duplicados_batch + existentes
    )

    return inseridos, duplicados


print("Funções prontas.")


In [ ]:
# ==========================
# 6. Importação
# ==========================

contagem_novas = {
    ano: 0
    for ano in ANOS
}

total_duplicadas = 0
total_sem_data = 0
total_processados = 0
total_erros = 0

with duckdb.connect(str(DB_PATH)) as con:

    for numero, caminho in enumerate(
        ficheiros,
        start=1,
    ):
        nome_ficheiro = Path(
            caminho
        ).name

        transacao_aberta = False

        try:
            cabecalho, linhas = (
                ler_linhas_xml_spreadsheet(
                    caminho
                )
            )

            novas_ficheiro = {
                ano: 0
                for ano in ANOS
            }

            duplicadas_ficheiro = 0
            sem_data_ficheiro = 0

            batches = {
                ano: []
                for ano in ANOS
            }

            for valores in linhas:

                if all(
                    valor is None
                    or str(valor).strip() == ""
                    for valor in valores
                ):
                    continue

                mapa = dict(
                    zip(
                        cabecalho,
                        valores,
                    )
                )

                ano = extrair_ano(
                    mapa
                )

                if ano not in ANOS:
                    sem_data_ficheiro += 1
                    continue

                batches[ano].append(
                    montar_linha(
                        mapa,
                        nome_ficheiro,
                    )
                )

            con.begin()
            transacao_aberta = True

            for ano in ANOS:
                inseridos, duplicados = (
                    inserir_batch(
                        con,
                        ano,
                        batches[ano],
                    )
                )

                novas_ficheiro[ano] += (
                    inseridos
                )

                duplicadas_ficheiro += (
                    duplicados
                )

            con.commit()
            transacao_aberta = False

            total_processados += 1

            for ano in ANOS:
                contagem_novas[ano] += (
                    novas_ficheiro[ano]
                )

            total_duplicadas += (
                duplicadas_ficheiro
            )

            total_sem_data += (
                sem_data_ficheiro
            )

            agora = datetime.now().isoformat(
                timespec="seconds"
            )

            print(
                f"[{numero}/{len(ficheiros)}] "
                f"{nome_ficheiro} -> "
                f"{novas_ficheiro['2025']} novas 2025, "
                f"{novas_ficheiro['2026']} novas 2026, "
                f"{duplicadas_ficheiro} já existiam, "
                f"{sem_data_ficheiro} sem data válida | "
                f"{agora}"
            )

        except Exception as erro:

            if transacao_aberta:
                con.rollback()

            total_erros += 1

            print(
                f"[{numero}/{len(ficheiros)}] "
                f"[ERRO] '{nome_ficheiro}': "
                f"{erro}"
            )

    con.checkpoint()


print("=" * 70)
print("RESUMO FINAL - ENTREGAS")
print("=" * 70)

print(
    f"Ficheiros encontrados:                 "
    f"{len(ficheiros)}"
)

print(
    f"Ficheiros processados:                 "
    f"{total_processados}"
)

print(
    f"Ficheiros com erro:                    "
    f"{total_erros}"
)

print(
    f"Linhas novas inseridas em 2025:        "
    f"{contagem_novas['2025']}"
)

print(
    f"Linhas novas inseridas em 2026:        "
    f"{contagem_novas['2026']}"
)

print(
    f"Linhas já existentes (não inseridas):  "
    f"{total_duplicadas}"
)

print(
    f"Linhas sem data válida:                "
    f"{total_sem_data}"
)

print("=" * 70)


In [ ]:
# ==========================
# 7. Validação ficheiros vs BD
# ==========================

def normalizar_chave(
    valores,
):
    return tuple(
        ""
        if valor is None
        else str(valor).strip()
        for valor in valores
    )


def obter_chaves_bd(con):
    chaves = {}

    colunas_sql = ", ".join(
        f'"{coluna}"'
        for coluna in CHAVE_UNICA
    )

    for ano in ANOS:

        linhas = con.execute(
            f"""
            SELECT {colunas_sql}
            FROM "entregas_{ano}"
            """
        ).fetchall()

        chaves[ano] = {
            normalizar_chave(
                linha
            )
            for linha in linhas
        }

    return chaves


def validar_ficheiros(
    ficheiros,
    chaves_bd,
):
    colunas_ordem = list(
        COLUNAS_ENTREGAS.keys()
    )

    chaves_ficheiros_por_mes = {}

    for caminho in ficheiros:

        cabecalho, linhas = (
            ler_linhas_xml_spreadsheet(
                caminho
            )
        )

        for valores in linhas:

            if all(
                valor is None
                or str(valor).strip() == ""
                for valor in valores
            ):
                continue

            mapa = dict(
                zip(
                    cabecalho,
                    valores,
                )
            )

            ano = extrair_ano(
                mapa
            )

            if ano not in ANOS:
                continue

            fecha = mapa.get(
                COLUNAS_ENTREGAS[
                    "FECHA_TEORICA_LLEGADA"
                ]
            )

            data_valida = validar_data(
                fecha
            )

            if data_valida is None:
                continue

            mes_ano = data_valida.strftime(
                "%m/%Y"
            )

            linha = montar_linha(
                mapa,
                Path(caminho).name,
            )

            chave = normalizar_chave(
                [
                    linha[
                        colunas_ordem.index(
                            coluna
                        )
                    ]
                    for coluna in CHAVE_UNICA
                ]
            )

            entrada = (
                chaves_ficheiros_por_mes
                .setdefault(
                    mes_ano,
                    {
                        "ano": ano,
                        "chaves": set(),
                    },
                )
            )

            entrada["chaves"].add(
                chave
            )

    resultado = []

    for mes_ano, dados in (
        chaves_ficheiros_por_mes.items()
    ):
        chaves_bd_ano = (
            chaves_bd[
                dados["ano"]
            ]
        )

        total = len(
            dados["chaves"]
        )

        na_bd = len(
            dados["chaves"]
            & chaves_bd_ano
        )

        resultado.append({
            "Mes/Ano": mes_ano,
            "Nos ficheiros": total,
            "Na BD": na_bd,
            "Em falta": total - na_bd,
        })

    df = pd.DataFrame(
        resultado
    )

    if df.empty:
        return df

    df["_ord"] = pd.to_datetime(
        df["Mes/Ano"],
        format="%m/%Y",
    )

    return (
        df
        .sort_values("_ord")
        .drop(columns="_ord")
        .reset_index(drop=True)
    )


with duckdb.connect(
    str(DB_PATH),
    read_only=True,
) as con:
    chaves_bd = obter_chaves_bd(
        con
    )

df_validacao_ficheiros = validar_ficheiros(
    ficheiros,
    chaves_bd,
)

df_validacao_ficheiros


In [ ]:
# ==========================
# 8. Diagnóstico agosto 2026
# ==========================

ficheiros_atual = listar_ficheiros_excel(
    PASTA_FICHEIROS
)

print(
    f"Ficheiros na pasta agora: "
    f"{len(ficheiros_atual)}"
)

for caminho in ficheiros_atual:
    print(
        " -",
        Path(caminho).name,
    )


print(
    "\nLinhas de 08/2026 por ficheiro:"
)

for caminho in ficheiros_atual:

    cabecalho, linhas = (
        ler_linhas_xml_spreadsheet(
            caminho
        )
    )

    idx_fecha = cabecalho.index(
        COLUNAS_ENTREGAS[
            "FECHA_TEORICA_LLEGADA"
        ]
    )

    n_agosto = sum(
        1
        for linha in linhas
        if (
            linha[idx_fecha]
            and "/08/2026"
            in str(linha[idx_fecha])
        )
    )

    if n_agosto:
        print(
            f"  {Path(caminho).name}: "
            f"{n_agosto}"
        )


with duckdb.connect(
    str(DB_PATH),
    read_only=True,
) as con:

    df_bd = con.execute("""
        SELECT
            FECHA_TEORICA_LLEGADA,
            ficheiro_origem,
            COUNT(*) AS n
        FROM entregas_2026
        WHERE FECHA_TEORICA_LLEGADA
              LIKE '%/08/2026'
        GROUP BY
            FECHA_TEORICA_LLEGADA,
            ficheiro_origem
        ORDER BY
            FECHA_TEORICA_LLEGADA,
            ficheiro_origem
    """).df()


print(
    "\nLinhas de 08/2026 já na BD:"
)

df_bd


In [ ]:
# ==========================
# 9. Validação de duplicados
# ==========================

with duckdb.connect(
    str(DB_PATH),
    read_only=True,
) as con:

    validacao = []
    duplicados = []

    for ano in ANOS:

        tabela = f"entregas_{ano}"

        total, ids_unicos = con.execute(f"""
            SELECT
                COUNT(*),
                COUNT(DISTINCT id)
            FROM "{tabela}"
        """).fetchone()

        grupo_sql = ", ".join(
            f'"{coluna}"'
            for coluna in CHAVE_UNICA
        )

        grupos_duplicados, linhas_duplicadas = (
            con.execute(f"""
                SELECT
                    COUNT(*),
                    COALESCE(SUM(n), 0)
                FROM (
                    SELECT
                        {grupo_sql},
                        COUNT(*) AS n
                    FROM "{tabela}"
                    GROUP BY
                        {grupo_sql}
                    HAVING COUNT(*) > 1
                )
            """).fetchone()
        )

        validacao.append({
            "Tabela": tabela,
            "Linhas": total,
            "IDs_Unicos": ids_unicos,
            "Grupos_Duplicados": grupos_duplicados,
            "Linhas_Duplicadas": linhas_duplicadas,
            "OK": (
                total == ids_unicos
                and grupos_duplicados == 0
            ),
        })

        if grupos_duplicados > 0:

            condicao = " AND ".join(
                f't."{coluna}" '
                f'IS NOT DISTINCT FROM '
                f'd."{coluna}"'
                for coluna in CHAVE_UNICA
            )

            df_dup = con.execute(f"""
                WITH chaves_duplicadas AS (
                    SELECT
                        {grupo_sql},
                        COUNT(*) AS qtd_duplicados
                    FROM "{tabela}"
                    GROUP BY
                        {grupo_sql}
                    HAVING COUNT(*) > 1
                )

                SELECT
                    d.qtd_duplicados,
                    t.*
                FROM "{tabela}" t
                INNER JOIN chaves_duplicadas d
                    ON {condicao}
                ORDER BY
                    TRY_CAST(t.id AS BIGINT)
            """).df()

            df_dup.insert(
                0,
                "Tabela",
                tabela,
            )

            duplicados.append(
                df_dup
            )


df_validacao = pd.DataFrame(
    validacao
)

df_duplicados = (
    pd.concat(
        duplicados,
        ignore_index=True,
    )
    if duplicados
    else pd.DataFrame()
)

df_validacao
